# Analyzing the internet through Twitter

In [39]:
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub
path = kagglehub.dataset_download("kazanova/sentiment140")
print("Path to dataset files:", path)

/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv
Path to dataset files: /kaggle/input/datasets/kazanova/sentiment140


In [40]:
# The file is not encoded in utf-8, using ISO-8859-1 instead
df = pd.read_csv('/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv', encoding='ISO-8859-1', header=None)

# Adding the column headers
df.columns = ['target', 'id', 'date', 'flag', 'user', 'text']

df.head()

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


It's now working with this special encoding, thanks to Sifat Muhammad Abdullah

In [41]:
df['target'].value_counts()

target
0    800000
4    800000
Name: count, dtype: int64

Missing the neutral input.

## Data Analysis

### Introduction

As we all know, Twitter is a gold mine of The Internet deepest (and shallowest) thoughts. People tends to let their minds pour out all of their ideas and views about the state of the world (and their world).

With this dataset, we can ask ourselves many questions like :
* How people felt at a specific date
* What are the most used words on the platform
* Who are the most chatty users

One of the issues is that their are only 5 available dates on this dataset, all of them ranging from May 22 to July 15 (2009).

## NLP Modeling

### Data Processing

In [42]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [43]:
import re

def clean_text(text):
    text = text.lower()

    # remove mentions
    text = re.sub(r'@\w+', '', text)

    # remove urls
    text = re.sub(r'http\S+|www\S+', '', text)

    # remove hashtags symbol only
    text = re.sub(r'#', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Converting the text to lowercase
df['text'] = df['text'].str.lower()

# Convert the text to str data type
df['text'] = df['text'].astype(str)

df['text'] = df['text'].apply(clean_text)

df['target'] = df['target'].map({
    0: 0,   # negative
    4: 1    # positive
})

#df.head()

df['target'].value_counts()

target
0    800000
1    800000
Name: count, dtype: int64

In [44]:
# Reducing the dataset size
df = (
    df.groupby('target', group_keys=False)
      .apply(lambda x: x.sample(50000, random_state=42))
      .reset_index(drop=True)
)

df['target'].value_counts()

/tmp/ipykernel_57/2785360951.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(50000, random_state=42))


target
0    50000
1    50000
Name: count, dtype: int64

In [45]:
# Tokenization
df['tokens'] = df['text'].apply(nltk.word_tokenize)

# Remove stopwords
stopwords = nltk.corpus.stopwords.words('english')
df['tokens'] = df['tokens'].apply(lambda x: [word for word in x if word not in stopwords])

### Model Creation

In [46]:
# Hyperparameters tuning
from sklearn.model_selection import GridSearchCV



In [47]:
X = df['text']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Converting text into numerical values.

In [48]:
"""vectorizer = TfidfVectorizer(max_features=5000)"""

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,3),
        max_features=50000,
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ('clf', LinearSVC(C=1.5))
])

#X_train_vectors = vectorizer.fit_transform(X_train)
#X_test_vectors = vectorizer.transform(X_test)

In [49]:
model = LinearSVC() # Might use LinearSVC to train faster

param_grid = {
    'clf__C': [0.01, 0.1, 1, 10]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidf',
                                        TfidfVectorizer(max_df=0.95,
                                                        max_features=50000,
                                                        min_df=2,
                                                        ngram_range=(1, 3),
                                                        sublinear_tf=True)),
                                       ('clf', LinearSVC(C=1.5))]),
             n_jobs=-1, param_grid={'clf__C': [0.01, 0.1, 1, 10]},
             scoring='accuracy')

In [50]:
print(grid.best_params_)
print(grid.best_score_)

{'clf__C': 0.1}
0.7881


In [51]:
best_model = grid.best_estimator_

predictions = best_model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, predictions))

print("Accuracy Score:", accuracy_score(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.79      0.79     10035
           1       0.79      0.80      0.79      9965

    accuracy                           0.79     20000
   macro avg       0.79      0.79      0.79     20000
weighted avg       0.79      0.79      0.79     20000

Accuracy Score: 0.7937


In [52]:
"""y_pred = model.predict(X_test_vectors)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Accuracy Score:", accuracy_score(y_test, y_pred))"""

'y_pred = model.predict(X_test_vectors)\n\nprint("Classification Report:")\nprint(classification_report(y_test, y_pred))\n\nprint("Accuracy Score:", accuracy_score(y_test, y_pred))'